# Notebook 06 — Train Mamba (SSM)

Selective State-Space Model for multi-site AirNow NO₂ forecasting (Gu & Dao, 2023).

**Input :** `(batch, seq_len, n_sites)` — normalised look-back window  
**Output :** `(batch, pred_len, n_sites)` — forecast horizon  
**Architecture :** `Linear → MambaBlock × n_layers [conv1d + selective SSM + gate] → Flatten → Linear`


In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

from data.load_airnow import load_sequences, DATA_DIR
from models.mamba_no2 import NO2Mamba
from models.transformer_no2 import _make_loader, evaluate

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
SEQ_LEN    = 24      # look-back window (hours)
PRED_LEN   = 6       # forecast horizon (hours)
D_MODEL    = 128     # state dimension
N_LAYERS   = 3       # Mamba blocks
D_STATE    = 16      # SSM state size
EXPAND     = 2       # inner-dim expansion factor
DROPOUT    = 0.1
EPOCHS     = 50
BATCH_SIZE = 64
LR         = 1e-3
PATIENCE   = 8
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Config : seq={SEQ_LEN}h  pred={PRED_LEN}h  d={D_MODEL}  layers={N_LAYERS}  d_state={D_STATE}")


## 1 · Load & Split Data


In [ ]:
print("Loading sequences …")
X, y, timestamps, site_codes = load_sequences(
    DATA_DIR, seq_len=SEQ_LEN, pred_len=PRED_LEN,
    stride=1, fill_nan=0.0, normalize=True,
)
N_SITES = X.shape[2]
n       = len(X)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

X_train, y_train = X[:n_train],              y[:n_train]
X_val,   y_val   = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test,  y_test  = X[n_train+n_val:],        y[n_train+n_val:]

print(f"  Sequences : {n:,}  (sites={N_SITES})")
print(f"  Train     : {n_train:,}  ({timestamps[0]} → {timestamps[n_train-1]})")
print(f"  Val       : {n_val:,}")
print(f"  Test      : {len(X_test):,}")


## 2 · Build Model


In [ ]:
model = NO2Mamba(
    n_sites=N_SITES, seq_len=SEQ_LEN, pred_len=PRED_LEN,
    d_model=D_MODEL, n_layers=N_LAYERS,
    d_state=D_STATE, expand=EXPAND, dropout=DROPOUT,
)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nParameters : {n_params:,}")

CKPT_NAME = f"mamba_s{SEQ_LEN}_p{PRED_LEN}_d{D_MODEL}.pt"
CKPT_PATH = OUTPUTS / CKPT_NAME
print(f"Checkpoint : {CKPT_PATH}")


## 3 · Train


In [ ]:
model = model.to(DEVICE)
opt     = torch.optim.Adam(model.parameters(), lr=LR)
sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
loss_fn = nn.MSELoss()
loader  = _make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)

history  = []
best_val = float("inf")
stale    = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(X_train)

    val_mse, val_mae = evaluate(model, X_val, y_val,
                                batch_size=BATCH_SIZE * 4, device=DEVICE)
    sched.step(val_mse)
    row = {"epoch": epoch, "train_mse": train_loss,
           "val_mse": val_mse, "val_mae": val_mae,
           "lr": opt.param_groups[0]["lr"]}
    history.append(row)

    marker = ""
    if val_mse < best_val:
        best_val = val_mse
        stale = 0
        torch.save(model.state_dict(), CKPT_PATH)
        marker = "  ✓ best"
    else:
        stale += 1

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"train={train_loss:.4f}  val_mse={val_mse:.4f}  val_mae={val_mae:.4f}  "
          f"lr={opt.param_groups[0]['lr']:.1e}  ({time.time()-t0:.1f}s){marker}")

    if stale >= PATIENCE:
        print(f"\nEarly stopping — best val_mse={best_val:.4f}")
        break

print(f"\nCheckpoint saved → {CKPT_PATH.name}")


## 4 · Evaluate on Test Set


In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
test_mse, test_mae = evaluate(model, X_test, y_test, device=DEVICE)
print(f"{'='*45}")
print(f"  TEST  MSE = {test_mse:.4f}   MAE = {test_mae:.4f}")
print(f"{'='*45}")

meta = {
    "model": "mamba", "n_sites": N_SITES,
    "seq_len": SEQ_LEN, "pred_len": PRED_LEN,
    "d_model": D_MODEL, "n_layers": N_LAYERS,
    "n_params": n_params,
    "test_mse": test_mse, "test_mae": test_mae,
    "history": history,
}
hist_path = OUTPUTS / CKPT_NAME.replace(".pt", "_history.json")
hist_path.write_text(json.dumps(meta, indent=2))
print(f"History saved → {hist_path.name}")


## 5 · Plots


In [ ]:
epochs_    = [h["epoch"]     for h in history]
train_mse_ = [h["train_mse"] for h in history]
val_mse_   = [h["val_mse"]   for h in history]
val_mae_   = [h["val_mae"]   for h in history]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs_, train_mse_, color="darkorange", label="Train MSE")
axes[0].plot(epochs_, val_mse_,   color="tomato",     label="Val MSE", ls="--")
axes[0].set_title("MSE over training"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_, val_mae_, color="darkorange")
axes[1].set_title("Validation MAE over training"); axes[1].set_xlabel("Epoch")
axes[1].grid(alpha=0.3)

fig.suptitle("Mamba — Training curves", fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUTS / "mamba_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Scatter ────────────────────────────────────────────────────────────────────
model.eval()
preds_list = []
with torch.no_grad():
    for xb, _ in _make_loader(X_test, y_test, 512, shuffle=False):
        preds_list.append(model(xb.to(DEVICE)).cpu().numpy())
preds  = np.concatenate(preds_list)
flat_y = y_test.ravel()
flat_p = preds.ravel()
idx    = np.random.choice(len(flat_y), min(8000, len(flat_y)), replace=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(flat_y[idx], flat_p[idx], alpha=0.15, s=5, color="darkorange")
lim = max(flat_y[idx].max(), flat_p[idx].max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=0.9, label="perfect")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("Actual NO₂ (normalised)")
ax.set_ylabel("Predicted NO₂ (normalised)")
ax.set_title(f"Mamba — Predicted vs Actual (test)\nMSE={test_mse:.4f}  MAE={test_mae:.4f}",
             fontweight="bold")
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(OUTPUTS / "mamba_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
